# LGN Converter — Live Colab GPU Worker

Select **Runtime → Change runtime type → T4 GPU**. This worker polls the secured Hostinger queue; it does not access YouTube directly. Keep the final cell running during a test session.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

assert torch.cuda.is_available(), 'Enable a T4 GPU first'
repo = Path('/content/lngconverter')
if (repo / '.git').exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/cubatic/lngconverter.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[inference,translation,worker,tts]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'demucs', 'git+https://github.com/huggingface/parler-tts.git'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'protobuf>=5.29,<7'], check=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Installation complete')


In [ ]:
import os
from getpass import getpass
from pathlib import Path

from lgnconverter.config import Settings
from lgnconverter.remote_worker import RemoteWorker

hf_token = getpass('Hugging Face read token: ').strip()
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
worker_key = getpass('Worker API key: ').strip()
assert worker_key, 'Worker key is required'
settings = Settings(
    whisper_model='small', whisper_device='cuda', whisper_compute_type='float16',
    enable_translation=True, translation_device='cuda',
    enable_tts=True, tts_device='cuda:0', enable_source_separation=True,
)
worker = RemoteWorker('https://lng.cubatic.co', worker_key, settings, Path('/content/lgn-worker'))
worker.run_forever()
